# Imports and Configuration

In [2]:

# Test basic PyTorch operations without loading BERT.
import torch

print("PyTorch:", torch.__version__, flush=True)

x = torch.randn(100, 100, requires_grad=True)
print("Tensor created.", flush=True)

loss = (x @ x).sum()
print("Forward pass completed.", flush=True)

loss.backward()
print("Backward pass completed.", flush=True)


PyTorch: 2.14.1
Tensor created.
Forward pass completed.
Backward pass completed.


In [3]:
# Install Hugging Face Datasets in the active Jupyter environment.
%pip install datasets

# Import libraries for data processing, machine learning, evaluation, visualization, and transformer models.

# General utilities
import os
import re
import json
import random
import warnings
from pathlib import Path
from collections import Counter

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data splitting and cross-validation
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    cross_val_predict,
    GridSearchCV
)

# Text feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Machine learning classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier

# Model pipelines
from sklearn.pipeline import Pipeline

# Classification evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    precision_recall_curve,
    average_precision_score,
    balanced_accuracy_score,
    make_scorer
)

# Model persistence
import joblib

# Transformer models (BERT / RoBERTa)
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset

%pip install -U accelerate

# Check installed transformer libraries and available computing hardware.
import importlib.util
import torch

for package in ["transformers", "datasets", "accelerate"]:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package}: {'Installed' if installed else 'Not installed'}")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")


# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
transformers: Installed
datasets: Installed
accelerate: Installed
PyTorch version: 2.14.1
CUDA available: False
MPS available: True


# Data Loading and Verification

In [4]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
print("Project directory:", PROJECT_DIR)
print("Data folders:", [p.name for p in (PROJECT_DIR / "data").iterdir()])

Project directory: /Users/Kashif/Library/CloudStorage/Box-Box/0.KB_NLP
Data folders: ['annotated', '.DS_Store', 'zzzArchive', '.ipynb_checkpoints', 'processed', 'raw']


In [5]:
# Define the folder containing annotated datasets.
ANNOTATED_DIR = PROJECT_DIR / "data" / "annotated"
for file in ANNOTATED_DIR.iterdir():
    if file.is_file() and not file.name.startswith("."):
        print(file.name)

class_imb1_200.xlsx
narr_ann.xlsx
narr_model.xlsx


In [6]:
# Load the manually annotated dataset for model training.
df = pd.read_excel(ANNOTATED_DIR / "narr_model.xlsx")

print(df.shape)

(580, 5)


In [7]:
# Check the distribution of survivor narrative labels.
print(df["survivor_narrative"].value_counts(dropna=False))

# Check for duplicate Reddit IDs and missing post text.
print("Duplicate IDs:", df["id"].duplicated().sum())
print("Missing text:", df["selftext"].isna().sum())

survivor_narrative
1    437
0    143
Name: count, dtype: int64
Duplicate IDs: 0
Missing text: 0


## Train-Test Split*** DO NOT RE-RUN
The manually annotated dataset (N = 580) was partitioned into training (70%) and testing (30%) subsets using stratified random sampling. Stratification was applied to preserve the distribution of survivor narrative labels across both subsets. A fixed random seed (42) was used to ensure reproducibility.

⚠️ *** **CONDITIONAL RERUN — FILE OUTPUT** ***

This cell writes data to disk and may overwrite existing output files. Rerun only when intentionally regenerating these outputs. Verify the source dataset and destination filenames before execution.

In [6]:
# Split the annotated dataset into stratified training (70%) and testing (30%) sets.
train_df, test_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["survivor_narrative"],
    random_state=42
)

print("Training:", train_df.shape)
print("Testing:", test_df.shape)

Training: (406, 5)
Testing: (174, 5)


In [11]:
# Verify the distribution of survivor narrative labels in each dataset.
print("Training labels:")
print(train_df["survivor_narrative"].value_counts())

print("\nTesting labels:")
print(test_df["survivor_narrative"].value_counts())

Training labels:
survivor_narrative
1    306
0    100
Name: count, dtype: int64

Testing labels:
survivor_narrative
1    131
0     43
Name: count, dtype: int64


In [12]:
# Save the stratified training and testing splits to the 'processed' data folder.
train_df.to_excel(PROJECT_DIR / "data/processed/narr_train_70.xlsx", index=False)
test_df.to_excel(PROJECT_DIR / "data/processed/narr_test_30.xlsx", index=False)

print("Training and testing datasets saved successfully.")

Training and testing datasets saved successfully.


⚠️ *** **TRAINING AND TEST SETS SAVED AS EXCEL FILES** ***

In [11]:

# Convert the saved Excel splits to CSV without overwriting existing files.
train_path = PROJECT_DIR / "data/processed/narr_train_70.csv"
test_path = PROJECT_DIR / "data/processed/narr_test_30.csv"

if train_path.exists() or test_path.exists():
    raise FileExistsError("One or both CSV files already exist.")

train_saved = pd.read_excel(PROJECT_DIR / "data/processed/narr_train_70.xlsx")
test_saved = pd.read_excel(PROJECT_DIR / "data/processed/narr_test_30.xlsx")

train_saved.to_csv(train_path, index=False, encoding="utf-8")
test_saved.to_csv(test_path, index=False, encoding="utf-8")

print("CSV files created successfully.")


CSV files created successfully.


⚠️ *** **TRAINING AND TEST SETS SAVED AS .CSV FILES** ***

## Modeling plan:
1. **Logistic regression:** Completed baseline, class weighting, and hyperparameter tuning.
2. **BERT:** Fine-tuning, five-fold stratified cross-validation, and evaluation.
3. **RoBERTa:** Repeat the same evaluation procedure.
4. **Model comparison:**  Compare all three approaches using consistent metrics.

# LOGISTIC REGRESSION MODEL DEVELOPMENT & OPTIMIZATION

In [7]:
# Load the existing CSV training and testing datasets.
train_df = pd.read_csv(PROJECT_DIR / "data/processed/narr_train_70.csv")
test_df = pd.read_csv(PROJECT_DIR / "data/processed/narr_test_30.csv")

print("Training:", train_df.shape)
print("Testing:", test_df.shape)

Training: (406, 5)
Testing: (174, 5)


In [8]:
# Build a TF-IDF and logistic regression pipeline for binary text classification.
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 1))),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

print(baseline)

Pipeline(steps=[('tfidf', TfidfVectorizer()),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])


This creates a pipeline with two components:
- TF-IDF: Converts Reddit posts into numerical features using individual words (unigrams).
- Logistic regression: Learns to classify posts as survivor narratives (1) or non-narratives (0).

In [9]:
# Combine post titles and bodies before training the baseline classifier.
X_train = (
    train_df["title"].fillna("") + " " +
    train_df["selftext"].fillna("")
).str.strip()

y_train = train_df["survivor_narrative"]

baseline.fit(X_train, y_train)

print("Baseline model trained successfully.")

Baseline model trained successfully.


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


## Cross-Validation
Five-fold stratified cross-validation was selected to balance computational efficiency and reliable performance estimation given the relatively small annotated training sample (n = 406). Stratification preserved the distribution of survivor narratives and non-narratives across folds, ensuring that both classes were represented during validation. Five folds also allowed each observation to contribute to validation once while maintaining approximately 80% of the development sample for training in each iteration.

In [10]:
# Evaluate baseline performance using five-fold stratified cross-validation.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "specificity": make_scorer(recall_score, pos_label=0),
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = cross_validate(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

for metric in scoring:
    scores = cv_results[f"test_{metric}"]
    print(f"{metric}: {scores.mean():.3f} (±{scores.std():.3f})")

accuracy: 0.793 (±0.013)
precision: 0.786 (±0.012)
recall: 0.997 (±0.007)
specificity: 0.170 (±0.068)
balanced_accuracy: 0.583 (±0.032)
f1: 0.879 (±0.006)
roc_auc: 0.933 (±0.028)


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


### *BASELINE Cross-Validation Results*

*The baseline TF-IDF and logistic regression classifier was evaluated using five-fold stratified cross-validation on the training dataset (n = 406). Model performance was assessed using accuracy, precision, recall, specificity, balanced accuracy, F1-score, and ROC-AUC.*

*The classifier achieved an accuracy of 0.793 (SD = 0.013), precision of 0.786 (SD = 0.012), recall of 0.997 (SD = 0.007), specificity of 0.170 (SD = 0.068), balanced accuracy of 0.583 (SD = 0.032), F1-score of 0.879 (SD = 0.006), and ROC-AUC of 0.933 (SD = 0.028).*

*Although recall and ROC-AUC were high, the substantially lower specificity indicated that the classifier frequently misclassified non-survivor narratives as survivor narratives. This pattern suggests that the default classification threshold and/or class imbalance may be contributing to poor minority-class performance. Consequently, subsequent experiments will evaluate strategies to improve specificity and balanced accuracy while maintaining adequate sensitivity.*

## Logistic Regression Optimization

In [11]:
# Evaluate class-weighted logistic regression using the same cross-validation folds.
weighted_lr = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 1))),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

weighted_results = cross_validate(
    weighted_lr,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

for metric in scoring:
    scores = weighted_results[f"test_{metric}"]
    print(f"{metric}: {scores.mean():.3f} (±{scores.std():.3f})")


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


accuracy: 0.882 (±0.045)
precision: 0.936 (±0.025)
recall: 0.905 (±0.049)
specificity: 0.810 (±0.073)
balanced_accuracy: 0.857 (±0.048)
f1: 0.920 (±0.031)
roc_auc: 0.934 (±0.027)


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


### *Class-Weighted Logistic Regression Results*

To address class imbalance in the annotated training dataset (n = 406; 306 survivor narratives and 100 non-survivor narratives), a class-weighted logistic regression classifier was evaluated using five-fold stratified cross-validation. The model retained the original TF-IDF unigram representation and logistic regression configuration, with the addition of class_weight="balanced". This parameter automatically adjusts class weights inversely proportional to class frequencies, assigning greater importance to errors involving the underrepresented non-survivor class.

Class weighting substantially improved specificity (0.170 to 0.810), balanced accuracy (0.583 to 0.857), precision (0.786 to 0.936), and F1-score (0.879 to 0.920). However, recall decreased from 0.997 to 0.905, indicating a tradeoff between correctly identifying non-survivor narratives and retaining genuine survivor narratives. ROC-AUC remained nearly unchanged (0.933 versus 0.934), suggesting that the models exhibited similar overall discriminatory ability across classification thresholds. The improvements in threshold-dependent metrics indicate that class weighting produced more balanced classification decisions at the default decision threshold.

These findings suggest that class imbalance contributed to the poor minority-class performance observed in the unweighted baseline. Class-weighted logistic regression therefore represents a stronger candidate for subsequent comparison with transformer-based classifiers. Final model selection will consider both false-positive and false-negative errors, with performance assessed on the held-out test dataset after model development.

## Logistic Regression Hyperparameter Tuning

In [12]:

# Tune weighted logistic regression and TF-IDF hyperparameters using stratified cross-validation.
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 5],
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    estimator=weighted_lr,
    param_grid=param_grid,
    scoring=scoring,
    refit="balanced_accuracy",
    cv=cv,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV balanced accuracy:", round(grid_search.best_score_, 3))


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.1

Best parameters: {'classifier__C': 1, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}
Best CV balanced accuracy: 0.871


In [13]:

# Display cross-validation metrics for the best hyperparameter configuration.
best_idx = grid_search.best_index_

for metric in scoring:
    mean = grid_search.cv_results_[f"mean_test_{metric}"][best_idx]
    std = grid_search.cv_results_[f"std_test_{metric}"][best_idx]
    print(f"{metric}: {mean:.3f} (±{std:.3f})")


accuracy: 0.887 (±0.040)
precision: 0.946 (±0.027)
recall: 0.902 (±0.042)
specificity: 0.840 (±0.080)
balanced_accuracy: 0.871 (±0.048)
f1: 0.923 (±0.027)
roc_auc: 0.937 (±0.023)


#### *Hyperparameter Tuning Results*

To further optimize the class-weighted logistic regression classifier, grid search was performed using five-fold stratified cross-validation on the training dataset (n = 406). Thirty hyperparameter configurations were evaluated by varying TF-IDF n-gram range (unigrams versus unigrams and bigrams), minimum document frequency (`min_df`: 1, 2, 5), and logistic regression regularization strength (`C`: 0.01, 0.1, 1, 10, 100). Balanced accuracy was specified as the primary optimization criterion, while accuracy, precision, recall, specificity, F1-score, and ROC-AUC were also evaluated.

The optimal configuration retained unigram TF-IDF features (`ngram_range=(1,1)`), increased the minimum document frequency to two (`min_df=2`), and retained the default regularization parameter (`C=1`). Class weighting remained balanced.

The tuned classifier achieved an accuracy of 0.887 (SD = 0.040), precision of 0.946 (SD = 0.027), recall of 0.902 (SD = 0.042), specificity of 0.840 (SD = 0.080), balanced accuracy of 0.871 (SD = 0.048), F1-score of 0.923 (SD = 0.027), and ROC-AUC of 0.937 (SD = 0.023).

Compared with the initial class-weighted model, hyperparameter tuning increased balanced accuracy from 0.857 to 0.871, specificity from 0.810 to 0.840, and precision from 0.936 to 0.946. Recall decreased marginally from 0.905 to 0.902. These findings suggest that excluding terms appearing in only one training document may improve classification performance while preserving sensitivity to survivor narratives.

Overall, the tuned class-weighted logistic regression classifier demonstrated promising performance for identifying first-person survivor narratives. However, because hyperparameters were selected using cross-validation performance, the reported results may be optimistic. Independent evaluation on the held-out test dataset will be necessary to assess generalizability.

# TRANSFORMER MODEL DEVELOPMENT & OPTIMIZATION — BERT

In [14]:

# Load the pretrained BERT tokenizer.
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Model: {MODEL_NAME}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Maximum sequence length: {tokenizer.model_max_length}")

Model: bert-base-uncased
Vocabulary size: 30,522
Maximum sequence length: 512


In [15]:
# Examine BERT token lengths in the training data.
token_lengths = [
    len(tokenizer.encode(text, add_special_tokens=True, truncation=False))
    for text in X_train
]

length_summary = pd.Series(token_lengths).describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

print(length_summary)
print(f"\nPosts exceeding 512 tokens: {sum(length > 512 for length in token_lengths)}")
print(f"Percentage exceeding 512 tokens: {np.mean(np.array(token_lengths) > 512):.1%}")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1043 > 512). Running this sequence through the model will result in indexing errors


count     406.000000
mean      385.916256
std       423.889816
min        11.000000
50%       256.000000
75%       479.250000
90%       843.500000
95%      1143.500000
99%      2119.050000
max      3627.000000
dtype: float64

Posts exceeding 512 tokens: 90
Percentage exceeding 512 tokens: 22.2%



#### *Token-Length Distribution and Truncation Considerations*

Token-length analysis was conducted on the 406 training observations using the pretrained `bert-base-uncased` tokenizer without truncation. The mean sequence length was 385.9 tokens (SD = 423.9), with a median of 256 tokens and a maximum of 3,627 tokens.

Of the 406 observations, 90 (22.2%) exceeded BERT's maximum input length of 512 tokens. Consequently, conventional BERT tokenization would truncate portions of these observations, potentially excluding information relevant to identifying first-person survivor narratives.

To establish a conventional transformer baseline, BERT will initially be fine-tuned using a maximum sequence length of 512 tokens. Subsequent analyses may evaluate chunk-based processing to preserve information from longer narratives and assess the potential effects of truncation on classification performance.

All original narrative texts will be retained without modification.


## Define five-fold stratified cross-validation

In [16]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train), start=1
):
    train_labels = y_train.iloc[train_idx]
    val_labels = y_train.iloc[val_idx]

    print(
        f"Fold {fold}: "
        f"Train={len(train_idx)}, "
        f"Validation={len(val_idx)}, "
        f"Validation positives={val_labels.sum()}, "
        f"Validation negatives={(val_labels == 0).sum()}"
    )

Fold 1: Train=324, Validation=82, Validation positives=62, Validation negatives=20
Fold 2: Train=325, Validation=81, Validation positives=61, Validation negatives=20
Fold 3: Train=325, Validation=81, Validation positives=61, Validation negatives=20
Fold 4: Train=325, Validation=81, Validation positives=61, Validation negatives=20
Fold 5: Train=325, Validation=81, Validation positives=61, Validation negatives=20


## Prepare the Hugging Face dataset

In [17]:
# Convert existing training observations into a Hugging Face dataset.
bert_df = pd.DataFrame({
    "text": X_train.to_numpy(),
    "labels": y_train.to_numpy().astype(int)
})

bert_dataset = Dataset.from_pandas(bert_df, preserve_index=False)

print(bert_dataset)
print("\nClass distribution:")
print(bert_df["labels"].value_counts())

Dataset({
    features: ['text', 'labels'],
    num_rows: 406
})

Class distribution:
labels
1    306
0    100
Name: count, dtype: int64


## Tokenization

In [18]:

# Tokenize training observations for BERT.
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512
    )

bert_tokenized = bert_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(bert_tokenized)


Map:   0%|          | 0/406 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 406
})


## BERT Model Initialization

In [19]:

# Initialize pretrained BERT for binary survivor-narrative classification.
bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(f"Model: {MODEL_NAME}")
print(f"Number of classes: {bert_model.config.num_labels}")
print(f"Total parameters: {bert_model.num_parameters():,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: bert-base-uncased
Number of classes: 2
Total parameters: 109,483,778


## BERT Fine-Tuning Configuration

In [20]:

# Define BERT fine-tuning hyperparameters.
BERT_LEARNING_RATE = 2e-5
BERT_BATCH_SIZE = 2
BERT_EPOCHS = 3
BERT_WEIGHT_DECAY = 0.01

print(f"Learning rate: {BERT_LEARNING_RATE}")
print(f"Batch size: {BERT_BATCH_SIZE}")
print(f"Epochs: {BERT_EPOCHS}")
print(f"Weight decay: {BERT_WEIGHT_DECAY}")


Learning rate: 2e-05
Batch size: 2
Epochs: 3
Weight decay: 0.01


In [21]:

# Define classification metrics for BERT cross-validation.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probabilities = torch.softmax(
        torch.tensor(logits), dim=1
    ).numpy()[:, 1]
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "specificity": recall_score(labels, predictions, pos_label=0, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, zero_division=0),
        "roc_auc": roc_auc_score(labels, probabilities)
    }


In [22]:

# Configure BERT training for five-fold cross-validation.
bert_training_args = TrainingArguments(
    output_dir="results/bert",
    learning_rate=BERT_LEARNING_RATE,
    per_device_train_batch_size=BERT_BATCH_SIZE,
    per_device_eval_batch_size=BERT_BATCH_SIZE,
    num_train_epochs=BERT_EPOCHS,
    weight_decay=BERT_WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    seed=42
)

print(f"Training device: {bert_training_args.device}")
print(f"Training batch size: {bert_training_args.per_device_train_batch_size}")
print(f"Epochs: {bert_training_args.num_train_epochs}")

Training device: mps
Training batch size: 2
Epochs: 3


## BERT Cross-Validation and Training

In [23]:

# Dynamically pad tokenized observations within each batch.
bert_data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("BERT data collator initialized.")


BERT data collator initialized.


In [24]:

# Prepare the same stratified cross-validation folds used for logistic regression.
bert_folds = list(cv.split(X_train, y_train))

for fold, (train_idx, val_idx) in enumerate(bert_folds, start=1):
    print(
        f"Fold {fold}: "
        f"{len(train_idx)} training observations, "
        f"{len(val_idx)} validation observations"
    )


Fold 1: 324 training observations, 82 validation observations
Fold 2: 325 training observations, 81 validation observations
Fold 3: 325 training observations, 81 validation observations
Fold 4: 325 training observations, 81 validation observations
Fold 5: 325 training observations, 81 validation observations


In [25]:

# Test one BERT forward and backward pass on the CPU.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to("cpu")

features = [
    {k: v for k, v in bert_tokenized[i].items() if k != "text"}
    for i in range(2)
]

batch = bert_data_collator(features)

model.train()
outputs = model(**batch)
outputs.loss.backward()

print(f"CPU training step completed. Loss: {outputs.loss.item():.4f}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


CPU training step completed. Loss: 0.6134


In [27]:

# Train and evaluate BERT using 5-fold stratified cross-validation.
import gc

bert_cv_results = []

# Remove any previously loaded models.
if "trainer" in globals():
    del trainer
if "model" in globals():
    del model
gc.collect()

if torch.backends.mps.is_available():
    torch.mps.empty_cache()

for fold, (train_idx, val_idx) in enumerate(bert_folds, start=1):
    print(f"\n--- BERT Fold {fold}/5 ---", flush=True)

    torch.manual_seed(42)

    fold_train = bert_tokenized.select(train_idx.tolist())
    fold_val = bert_tokenized.select(val_idx.tolist())

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    trainer = Trainer(
        model=model,
        args=bert_training_args,
        train_dataset=fold_train,
        eval_dataset=fold_val,
        data_collator=bert_data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()
    metrics = trainer.evaluate()

    bert_cv_results.append({
        "fold": fold,
        **{
            k.replace("eval_", ""): v
            for k, v in metrics.items()
            if k.startswith("eval_")
        }
    })

    # Save results after each fold.
    bert_cv_df = pd.DataFrame(bert_cv_results)
    bert_cv_df.to_csv("results/bert_cv_results.csv", index=False)

    print(f"Fold {fold} complete.", flush=True)

    # Release memory before the next fold.
    del trainer, model
    gc.collect()

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

# Calculate mean and standard deviation across folds.
metric_cols = [
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "balanced_accuracy",
    "f1",
    "roc_auc"
]

bert_summary = bert_cv_df[metric_cols].agg(["mean", "std"]).T

print("\nBERT 5-Fold Cross-Validation Results:")
for metric, row in bert_summary.iterrows():
    print(f"{metric}: {row['mean']:.3f} ± {row['std']:.3f}")



--- BERT Fold 1/5 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/opt/anaconda3/lib/pyt

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
1,0.650655,0.370916,0.865854,0.859155,0.983871,0.500000,0.741935,0.917293,0.919355
2,0.532120,0.391834,0.902439,0.909091,0.967742,0.700000,0.833871,0.937500,0.928226
3,0.437402,0.410101,0.902439,0.909091,0.967742,0.700000,0.833871,0.937500,0.934677


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
0.437402,0.410101,3,0.902439,0.909091,0.967742,0.700000,0.833871,0.937500,0.934677


... (21 outputs left out)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
1,0.598179,0.458142,0.864198,0.847222,1.000000,0.450000,0.725000,0.917293,0.913115
2,0.365229,0.171517,0.962963,0.983333,0.967213,0.950000,0.958607,0.975207,0.986885
3,0.154840,0.234142,0.962963,0.983333,0.967213,0.950000,0.958607,0.975207,0.970492


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
0.154840,0.234142,3,0.962963,0.983333,0.967213,0.950000,0.958607,0.975207,0.970492


Fold 5 complete.

BERT 5-Fold Cross-Validation Results:
accuracy: 0.909 ± 0.038
precision: 0.928 ± 0.033
recall: 0.954 ± 0.036
specificity: 0.770 ± 0.110
balanced_accuracy: 0.862 ± 0.057
f1: 0.940 ± 0.025
roc_auc: 0.953 ± 0.013


## BERT Hyperparameter Tuning

In [28]:

# BERT hyperparameter configurations
from itertools import product

bert_param_grid = [
    {"learning_rate": lr, "num_train_epochs": epochs}
    for lr, epochs in product(
        [1e-5, 2e-5, 3e-5],
        [2, 3]
    )
]

bert_tuning_grid = pd.DataFrame(bert_param_grid)
bert_tuning_grid


,learning_rate,num_train_epochs
0,0.00001,2
1,0.00001,3
2,0.00002,2
3,0.00002,3
4,0.00003,2
5,0.00003,3


Following baseline evaluation, BERT was subjected to hyperparameter optimization to assess whether adjustments to the training configuration improved classification performance. A grid search was conducted across three learning rates (1 × 10⁻⁵, 2 × 10⁻⁵, and 3 × 10⁻⁵) and two training durations (2 and 3 epochs), yielding six configurations.

Batch size (2) and weight decay (0.01) were held constant to maintain comparability with the baseline model and accommodate available computational resources. Each configuration was evaluated using the same five-fold stratified cross-validation partitions established during baseline evaluation.

Model selection was guided by a target precision of ≥0.95 and a minimum recall of ≥0.90. Among configurations satisfying both criteria, balanced accuracy was used to determine the preferred model.


In [29]:

# Training arguments for BERT hyperparameter tuning
def make_bert_tuning_args(learning_rate, num_train_epochs):
    return TrainingArguments(
        output_dir="results/bert_tuning",
        learning_rate=learning_rate,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        report_to="none",
        seed=42
    )

#### LONG RUN TIME ⚠️⚠️⚠️

In [30]:

# Five-fold cross-validation for BERT hyperparameter tuning
import gc
import os

os.makedirs("results", exist_ok=True)

results_path = "results/bert_tuning_cv_results.csv"
baseline_lr = 2e-5
baseline_epochs = 3

# Resume completed folds if results already exist
if os.path.exists(results_path):
    tuning_results = pd.read_csv(results_path).to_dict("records")
else:
    tuning_results = []

# Remove previous model objects
if "trainer" in globals():
    del trainer
if "model" in globals():
    del model

gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

for params in bert_param_grid:
    lr = params["learning_rate"]
    epochs = params["num_train_epochs"]

    # Baseline configuration was already evaluated
    if lr == baseline_lr and epochs == baseline_epochs:
        print("Skipping previously evaluated baseline BERT.")
        continue

    for fold, (train_idx, val_idx) in enumerate(bert_folds, start=1):

        # Avoid repeating completed folds
        completed = any(
            abs(r["learning_rate"] - lr) < 1e-10
            and r["num_train_epochs"] == epochs
            and r["fold"] == fold
            for r in tuning_results
        )

        if completed:
            continue

        print(
            f"\nBERT tuning: LR={lr}, epochs={epochs}, "
            f"fold={fold}/5",
            flush=True
        )

        torch.manual_seed(42)

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=2
        )

        trainer = Trainer(
            model=model,
            args=make_bert_tuning_args(lr, epochs),
            train_dataset=bert_tokenized.select(train_idx.tolist()),
            eval_dataset=bert_tokenized.select(val_idx.tolist()),
            data_collator=bert_data_collator,
            compute_metrics=compute_metrics
        )

        trainer.train()
        metrics = trainer.evaluate()

        tuning_results.append({
            "learning_rate": lr,
            "num_train_epochs": epochs,
            "fold": fold,
            **{
                k.replace("eval_", ""): v
                for k, v in metrics.items()
                if k.startswith("eval_")
            }
        })

        # Save results after every completed fold
        pd.DataFrame(tuning_results).to_csv(
            results_path, index=False
        )

        print(f"Fold {fold} saved.", flush=True)

        del trainer, model
        gc.collect()

        if torch.backends.mps.is_available():
            torch.mps.empty_cache()

print("\nBERT hyperparameter tuning complete.")



BERT tuning: LR=1e-05, epochs=2, fold=1/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/opt/anaconda3/lib/pyt

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
1,0.588615,0.458204,0.756098,0.756098,1.000000,0.000000,0.500000,0.861111,0.933871
2,0.488041,0.412812,0.804878,0.794872,1.000000,0.200000,0.600000,0.885714,0.947581


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
0.488041,0.412812,2,0.804878,0.794872,1.000000,0.200000,0.600000,0.885714,0.947581


... (141 outputs left out)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
1,0.666574,0.372851,0.876543,0.947368,0.885246,0.850000,0.867623,0.915254,0.963934
2,0.358731,0.312424,0.913580,0.909091,0.983607,0.700000,0.841803,0.944882,0.977869
3,0.145944,0.181917,0.962963,0.967742,0.983607,0.900000,0.941803,0.975610,0.977869


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
0.145944,0.181917,3,0.962963,0.967742,0.983607,0.900000,0.941803,0.975610,0.977869


Fold 5 saved.

BERT hyperparameter tuning complete.


# TRANSFORMER MODEL DEVELOPMENT & OPTIMIZATION — RoBERTa

### RoBERTa: Baseline Model

RoBERTa (Robustly Optimized BERT Pretraining Approach) was evaluated as an alternative transformer-based classifier for identifying first-person survivor narratives. The pretrained `roberta-base` model was selected to facilitate comparison with BERT using a similarly sized transformer architecture. The same stratified five-fold cross-validation partitions and evaluation metrics were retained to ensure consistency across models. Baseline hyperparameters were initially held consistent with those used for baseline BERT, including a learning rate of 2 × 10⁻⁵, three training epochs, a batch size of 2, and weight decay of 0.01.

In [31]:

# Initialize RoBERTa tokenizer and tokenize training data
ROBERTA_MODEL = "roberta-base"

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL)

def tokenize_roberta(batch):
    return roberta_tokenizer(
        batch["text"],
        truncation=True,
        max_length=512
    )

roberta_tokenized = bert_dataset.map(
    tokenize_roberta,
    batched=True,
    remove_columns=["text"]
)

roberta_data_collator = DataCollatorWithPadding(
    tokenizer=roberta_tokenizer
)


Map:   0%|          | 0/406 [00:00<?, ? examples/s]

In [32]:

# Baseline RoBERTa training parameters
ROBERTA_LEARNING_RATE = 2e-5
ROBERTA_BATCH_SIZE = 2
ROBERTA_EPOCHS = 3
ROBERTA_WEIGHT_DECAY = 0.01

roberta_training_args = TrainingArguments(
    output_dir="results/roberta",
    learning_rate=ROBERTA_LEARNING_RATE,
    per_device_train_batch_size=ROBERTA_BATCH_SIZE,
    per_device_eval_batch_size=ROBERTA_BATCH_SIZE,
    num_train_epochs=ROBERTA_EPOCHS,
    weight_decay=ROBERTA_WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    seed=42
)


## Five-Fold Cross-Validation

In [33]:

# Five-fold cross-validation for baseline RoBERTa
import gc

roberta_cv_results = []

for fold, (train_idx, val_idx) in enumerate(bert_folds, start=1):
    print(f"RoBERTa: Fold {fold}/5")

    torch.manual_seed(42)

    model = AutoModelForSequenceClassification.from_pretrained(
        ROBERTA_MODEL,
        num_labels=2
    )

    trainer = Trainer(
        model=model,
        args=roberta_training_args,
        train_dataset=roberta_tokenized.select(train_idx.tolist()),
        eval_dataset=roberta_tokenized.select(val_idx.tolist()),
        data_collator=roberta_data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()
    metrics = trainer.evaluate()

    roberta_cv_results.append({
        "fold": fold,
        **{k.replace("eval_", ""): v for k, v in metrics.items()
           if k.startswith("eval_")}
    })

    del trainer, model
    gc.collect()

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

roberta_cv_df = pd.DataFrame(roberta_cv_results)
roberta_cv_df.to_csv("results/roberta_cv_results.csv", index=False)


RoBERTa: Fold 1/5


model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  su

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
1,0.707945,0.273842,0.914634,0.923077,0.967742,0.750000,0.858871,0.944882,0.956452
2,0.325270,0.476483,0.902439,0.921875,0.951613,0.750000,0.850806,0.936508,0.954839
3,0.192936,0.589527,0.890244,0.907692,0.951613,0.700000,0.825806,0.929134,0.963710


... (22 outputs left out)

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  su

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
1,0.773865,0.260806,0.864198,0.847222,1.000000,0.450000,0.725000,0.917293,0.977049
2,0.327705,0.176140,0.962963,0.953125,1.000000,0.850000,0.925000,0.976000,0.990984
3,0.165542,0.099224,0.987654,0.983871,1.000000,0.950000,0.975000,0.991870,0.990984


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
0.165542,0.099224,3,0.987654,0.983871,1.000000,0.950000,0.975000,0.991870,0.990984



### RoBERTa Hyperparameter Tuning

A grid search was conducted to evaluate learning rates of 1e-5, 2e-5, and 3e-5 across two and three training epochs. Each configuration was evaluated using the same five stratified folds employed in the baseline models. The baseline configuration (learning rate = 2e-5; epochs = 3) was not retrained. Model performance was evaluated using accuracy, precision, recall, specificity, balanced accuracy, F1-score, and ROC-AUC.


In [34]:

import gc
from pathlib import Path

roberta_grid = [
    (1e-5, 2),
    (1e-5, 3),
    (2e-5, 2),
    (3e-5, 2),
    (3e-5, 3)
]

results_path = Path("results/roberta_tuning_cv_results.csv")
roberta_tuning_results = []

for lr, epochs in roberta_grid:
    for fold, (train_idx, val_idx) in enumerate(bert_folds, start=1):
        print(f"RoBERTa | LR={lr} | Epochs={epochs} | Fold {fold}/5")

        torch.manual_seed(42)

        model = AutoModelForSequenceClassification.from_pretrained(
            ROBERTA_MODEL,
            num_labels=2
        )

        args = TrainingArguments(
            output_dir="results/roberta_tuning",
            learning_rate=lr,
            per_device_train_batch_size=ROBERTA_BATCH_SIZE,
            per_device_eval_batch_size=ROBERTA_BATCH_SIZE,
            num_train_epochs=epochs,
            weight_decay=ROBERTA_WEIGHT_DECAY,
            eval_strategy="epoch",
            save_strategy="no",
            logging_strategy="epoch",
            report_to="none",
            seed=42
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=roberta_tokenized.select(train_idx.tolist()),
            eval_dataset=roberta_tokenized.select(val_idx.tolist()),
            data_collator=roberta_data_collator,
            compute_metrics=compute_metrics
        )

        trainer.train()
        metrics = trainer.evaluate()

        roberta_tuning_results.append({
            "learning_rate": lr,
            "epochs": epochs,
            "fold": fold,
            **{
                k.replace("eval_", ""): v
                for k, v in metrics.items()
                if k.startswith("eval_")
            }
        })

        pd.DataFrame(roberta_tuning_results).to_csv(
            results_path, index=False
        )

        del trainer, model
        gc.collect()

        if torch.backends.mps.is_available():
            torch.mps.empty_cache()

print("RoBERTa hyperparameter tuning complete.")


RoBERTa | LR=1e-05 | Epochs=2 | Fold 1/5


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  su

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
1,0.781129,0.515246,0.756098,0.756098,1.000000,0.000000,0.500000,0.861111,0.939516
2,0.489082,0.376132,0.756098,0.756098,1.000000,0.000000,0.500000,0.861111,0.961290


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
0.489082,0.376132,2,0.756098,0.756098,1.000000,0.000000,0.500000,0.861111,0.961290


... (141 outputs left out)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
1,0.739115,0.249941,0.913580,0.982143,0.901639,0.950000,0.925820,0.940171,0.962295
2,0.425026,0.248391,0.962963,0.953125,1.000000,0.850000,0.925000,0.976000,0.981967
3,0.190915,0.146035,0.975309,0.983607,0.983607,0.950000,0.966803,0.983607,0.980328


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,Specificity,Balanced Accuracy,F1,Roc Auc
0.190915,0.146035,3,0.975309,0.983607,0.983607,0.950000,0.966803,0.983607,0.980328


RoBERTa hyperparameter tuning complete.
